# Train Two Hoeffding Adaptive Tree Models for the Raspberry Pi

This notebook trains:

1. **Environmental health HAT**
   - Dataset: `agriculture_dataset`
   - Inputs: temperature, humidity, rainfall, wind speed, soil moisture, and soil pH
   - Output: `Healthy` or `Unhealthy`

2. **Nutrient-inclusive plant stress HAT**
   - Dataset: `plant_health_data`
   - Inputs: soil moisture, ambient temperature, soil temperature, humidity, light intensity, soil pH, nitrogen, phosphorus, and potassium
   - Output: `Healthy`, `Moderate Stress`, or `High Stress`

## Important scientific limitation

The first dataset's label is `Crop_Health_Label`, not a verified cause-specific environmental-stress label. The second dataset's label is overall plant-health status, not a laboratory-confirmed nutrient-deficiency diagnosis. Therefore, these are best described as an **environmental-feature health branch** and a **nutrient-inclusive stress branch**.

The notebook uses an 80/20 stratified split. The HAT first learns from the 80% development stream. It then evaluates the remaining 20% in **test-then-train** order: predict first, measure performance, and only then learn the observation.


In [ ]:
# Cell 1 - Mount Google Drive and install dependencies

from google.colab import drive
drive.mount("/content/drive")

!pip install -q river==0.22.0 scikit-learn pandas matplotlib


In [ ]:
# Cell 2 - Imports and reproducibility

import json
import pickle
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from river import metrics, tree

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("River version:", version("river"))
print("Pandas version:", pd.__version__)


In [ ]:
# Cell 3 - Locate the two CSV files in Google Drive

PROJECT_DIR = Path("/content/drive/MyDrive/smart_farming_project")

def locate_csv(project_dir, candidate_names, required_columns):
    for name in candidate_names:
        candidate = project_dir / name
        if candidate.exists():
            columns = set(pd.read_csv(candidate, nrows=0).columns)
            if set(required_columns).issubset(columns):
                return candidate

    for candidate in project_dir.rglob("*.csv"):
        try:
            columns = set(pd.read_csv(candidate, nrows=0).columns)
        except Exception:
            continue

        if set(required_columns).issubset(columns):
            return candidate

    raise FileNotFoundError(
        f"Could not find a CSV containing all required columns: {required_columns}"
    )

ENV_REQUIRED = [
    "Temperature", "Humidity", "Rainfall", "Wind_Speed",
    "Soil_Moisture", "Soil_pH", "Crop_Health_Label",
]

NUTRIENT_REQUIRED = [
    "Soil_Moisture", "Ambient_Temperature", "Soil_Temperature",
    "Humidity", "Light_Intensity", "Soil_pH", "Nitrogen_Level",
    "Phosphorus_Level", "Potassium_Level", "Plant_Health_Status",
]

agriculture_file = locate_csv(
    PROJECT_DIR,
    [
        "agriculture_dataset.csv",
        "agriculture_dataset(2).csv",
        "agriculture_dataset_with_matched_npk.csv",
    ],
    ENV_REQUIRED,
)

plant_health_file = locate_csv(
    PROJECT_DIR,
    ["plant_health_data.csv", "plant_health_data(2).csv"],
    NUTRIENT_REQUIRED,
)

print("Agriculture dataset:", agriculture_file)
print("Plant-health dataset:", plant_health_file)


In [ ]:
# Cell 4 - Load and inspect the datasets

agriculture_df = pd.read_csv(agriculture_file)
plant_health_df = pd.read_csv(plant_health_file)

print("Agriculture shape:", agriculture_df.shape)
print("Plant-health shape:", plant_health_df.shape)

print("\nAgriculture target distribution:")
print(agriculture_df["Crop_Health_Label"].value_counts(dropna=False).sort_index())

print("\nPlant-health target distribution:")
print(plant_health_df["Plant_Health_Status"].value_counts(dropna=False))

if "Crop_Type" in agriculture_df.columns:
    print("\nCrop types represented by the agriculture dataset:")
    print(agriculture_df["Crop_Type"].value_counts(dropna=False))


## Feature definitions

Only sensor-compatible columns are used. Image-derived features such as NDVI, SAVI, thermal images, segmentation, bounding boxes, and crop-stress indicators are intentionally excluded.


In [ ]:
# Cell 5 - Define exact features and clean the data

ENV_FEATURES = [
    "Temperature",
    "Humidity",
    "Rainfall",
    "Wind_Speed",
    "Soil_Moisture",
    "Soil_pH",
]
ENV_TARGET = "Crop_Health_Label"

NUTRIENT_FEATURES = [
    "Soil_Moisture",
    "Ambient_Temperature",
    "Soil_Temperature",
    "Humidity",
    "Light_Intensity",
    "Soil_pH",
    "Nitrogen_Level",
    "Phosphorus_Level",
    "Potassium_Level",
]
NUTRIENT_TARGET = "Plant_Health_Status"


def prepare_numeric_classification_data(df, features, target, label_mapper=None):
    data = df[features + [target]].copy()

    for feature in features:
        data[feature] = pd.to_numeric(data[feature], errors="coerce")

    data = data.dropna(subset=features + [target]).reset_index(drop=True)

    if label_mapper is not None:
        data["_label"] = data[target].map(label_mapper)
        if data["_label"].isna().any():
            unknown = data.loc[data["_label"].isna(), target].unique().tolist()
            raise ValueError(f"Unmapped target labels: {unknown}")
    else:
        data["_label"] = data[target].astype(str).str.strip()

    return data[features], data["_label"]


ENV_LABEL_MAP = {
    1: "Healthy",
    0: "Unhealthy",
    "1": "Healthy",
    "0": "Unhealthy",
}

X_env, y_env = prepare_numeric_classification_data(
    agriculture_df,
    ENV_FEATURES,
    ENV_TARGET,
    ENV_LABEL_MAP,
)

X_nutrient, y_nutrient = prepare_numeric_classification_data(
    plant_health_df,
    NUTRIENT_FEATURES,
    NUTRIENT_TARGET,
)

expected_nutrient_labels = {"Healthy", "Moderate Stress", "High Stress"}
actual_nutrient_labels = set(y_nutrient.unique())

if actual_nutrient_labels != expected_nutrient_labels:
    raise ValueError(
        "Unexpected plant-health labels. "
        f"Expected {expected_nutrient_labels}, found {actual_nutrient_labels}"
    )

print("Environmental rows after cleaning:", len(X_env))
print(y_env.value_counts())

print("\nNutrient-inclusive rows after cleaning:", len(X_nutrient))
print(y_nutrient.value_counts())


In [ ]:
# Cell 6 - HAT factory and reusable streaming helpers

def make_hat(seed=RANDOM_SEED):
    return tree.HoeffdingAdaptiveTreeClassifier(
        grace_period=200,
        max_depth=20,
        split_criterion="info_gain",
        delta=1e-7,
        tau=0.05,
        leaf_prediction="nba",
        nb_threshold=0,
        bootstrap_sampling=True,
        max_size=64.0,
        memory_estimate_period=10_000,
        remove_poor_attrs=True,
        seed=seed,
    )


def compute_balanced_class_weights(y_train):
    counts = Counter(y_train)
    total = len(y_train)
    classes = len(counts)

    return {
        label: total / (classes * count)
        for label, count in counts.items()
    }


def iter_rows(X, y):
    columns = list(X.columns)

    for values, label in zip(
        X.itertuples(index=False, name=None),
        y.tolist(),
    ):
        x = {
            column: float(value)
            for column, value in zip(columns, values)
        }
        yield x, label


def split_stream_data(X, y, test_size=0.20):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=RANDOM_SEED,
        stratify=y,
        shuffle=True,
    )

    train_order = np.random.default_rng(RANDOM_SEED).permutation(len(X_train))
    test_order = np.random.default_rng(RANDOM_SEED + 1).permutation(len(X_test))

    X_train = X_train.iloc[train_order].reset_index(drop=True)
    y_train = y_train.iloc[train_order].reset_index(drop=True)

    X_test = X_test.iloc[test_order].reset_index(drop=True)
    y_test = y_test.iloc[test_order].reset_index(drop=True)

    return X_train, X_test, y_train, y_test


In [ ]:
# Cell 7 - Train and progressively evaluate one HAT

def train_and_progressively_evaluate(
    X,
    y,
    task_name,
    use_class_weights=True,
):
    X_train, X_test, y_train, y_test = split_stream_data(X, y)

    model = make_hat()
    class_weights = compute_balanced_class_weights(y_train)

    print("=" * 80)
    print(task_name)
    print("=" * 80)
    print("Development stream:", len(X_train))
    print("Progressive evaluation stream:", len(X_test))
    print("Training class weights:", class_weights)

    for x, label in iter_rows(X_train, y_train):
        weight = class_weights[label] if use_class_weights else 1.0
        model.learn_one(x, label, w=weight)

    accuracy = metrics.Accuracy()
    balanced_accuracy = metrics.BalancedAccuracy()
    macro_f1 = metrics.MacroF1()
    report = metrics.ClassificationReport()
    confusion = metrics.ConfusionMatrix()

    history = []
    history_step = max(1, len(X_test) // 100)

    for step, (x, true_label) in enumerate(iter_rows(X_test, y_test), start=1):
        predicted_label = model.predict_one(x)

        if predicted_label is not None:
            accuracy.update(true_label, predicted_label)
            balanced_accuracy.update(true_label, predicted_label)
            macro_f1.update(true_label, predicted_label)
            report.update(true_label, predicted_label)
            confusion.update(true_label, predicted_label)

        weight = class_weights.get(true_label, 1.0) if use_class_weights else 1.0
        model.learn_one(x, true_label, w=weight)

        if step % history_step == 0 or step == len(X_test):
            history.append({
                "step": step,
                "accuracy": float(accuracy.get()),
                "balanced_accuracy": float(balanced_accuracy.get()),
                "macro_f1": float(macro_f1.get()),
            })

    results = {
        "task": task_name,
        "development_rows": len(X_train),
        "progressive_test_rows": len(X_test),
        "accuracy": float(accuracy.get()),
        "balanced_accuracy": float(balanced_accuracy.get()),
        "macro_f1": float(macro_f1.get()),
        "classification_report": str(report),
        "confusion_matrix": str(confusion),
        "class_weights": class_weights,
        "tree": {
            "height": getattr(model, "height", None),
            "n_nodes": getattr(model, "n_nodes", None),
            "n_branches": getattr(model, "n_branches", None),
            "n_leaves": getattr(model, "n_leaves", None),
            "n_active_leaves": getattr(model, "n_active_leaves", None),
            "n_alternate_trees": getattr(model, "n_alternate_trees", None),
            "n_switch_alternate_trees": getattr(model, "n_switch_alternate_trees", None),
        },
    }

    print("\nFinal progressive metrics")
    print(f"Accuracy:          {results['accuracy']:.4f}")
    print(f"Balanced accuracy: {results['balanced_accuracy']:.4f}")
    print(f"Macro F1:          {results['macro_f1']:.4f}")

    print("\nClassification report")
    print(report)

    print("\nConfusion matrix")
    print(confusion)

    print("\nTree statistics")
    print(json.dumps(results["tree"], indent=2))

    return model, results, pd.DataFrame(history)


In [ ]:
# Cell 8 - Train Model 1: environmental-feature health HAT

environmental_hat, environmental_results, environmental_history = (
    train_and_progressively_evaluate(
        X_env,
        y_env,
        task_name="Environmental-feature crop-health HAT",
        use_class_weights=True,
    )
)


In [ ]:
# Cell 9 - Plot Model 1 progressive metrics

plt.figure(figsize=(9, 5))
plt.plot(environmental_history["step"], environmental_history["accuracy"], label="Accuracy")
plt.plot(environmental_history["step"], environmental_history["balanced_accuracy"], label="Balanced accuracy")
plt.plot(environmental_history["step"], environmental_history["macro_f1"], label="Macro F1")
plt.xlabel("Progressive evaluation observations")
plt.ylabel("Score")
plt.title("Environmental-feature HAT performance")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Cell 10 - Train Model 2: nutrient-inclusive stress HAT

nutrient_hat, nutrient_results, nutrient_history = (
    train_and_progressively_evaluate(
        X_nutrient,
        y_nutrient,
        task_name="Nutrient-inclusive plant-stress HAT",
        use_class_weights=True,
    )
)


In [ ]:
# Cell 11 - Plot Model 2 progressive metrics

plt.figure(figsize=(9, 5))
plt.plot(nutrient_history["step"], nutrient_history["accuracy"], label="Accuracy")
plt.plot(nutrient_history["step"], nutrient_history["balanced_accuracy"], label="Balanced accuracy")
plt.plot(nutrient_history["step"], nutrient_history["macro_f1"], label="Macro F1")
plt.xlabel("Progressive evaluation observations")
plt.ylabel("Score")
plt.title("Nutrient-inclusive HAT performance")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Cell 12 - Save both trained models and their metadata to Google Drive

OUTPUT_DIR = PROJECT_DIR / "raspberry_pi_hat_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

environmental_model_path = OUTPUT_DIR / "environmental_health_hat.pkl"
nutrient_model_path = OUTPUT_DIR / "nutrient_stress_hat.pkl"
metadata_path = OUTPUT_DIR / "hat_model_metadata.json"

with environmental_model_path.open("wb") as file:
    pickle.dump(environmental_hat, file)

with nutrient_model_path.open("wb") as file:
    pickle.dump(nutrient_hat, file)

metadata = {
    "river_version": version("river"),
    "random_seed": RANDOM_SEED,
    "evaluation": "80% pretraining followed by 20% test-then-train progressive evaluation",
    "environmental_model": {
        "file": environmental_model_path.name,
        "source_dataset": str(agriculture_file),
        "features": ENV_FEATURES,
        "target": ENV_TARGET,
        "labels": sorted(y_env.unique().tolist()),
        "results": environmental_results,
        "limitation": (
            "Crop_Health_Label is a general crop-health label. "
            "It does not prove that environmental variables caused the unhealthy state."
        ),
    },
    "nutrient_model": {
        "file": nutrient_model_path.name,
        "source_dataset": str(plant_health_file),
        "features": NUTRIENT_FEATURES,
        "target": NUTRIENT_TARGET,
        "labels": sorted(y_nutrient.unique().tolist()),
        "results": nutrient_results,
        "limitation": (
            "Plant_Health_Status is an overall stress label. "
            "It is not a cause-specific nutrient-deficiency diagnosis."
        ),
    },
}

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, default=str)

print("Saved:")
print(environmental_model_path)
print(nutrient_model_path)
print(metadata_path)


In [ ]:
# Cell 13 - Reload the saved models and verify predictions

with environmental_model_path.open("rb") as file:
    loaded_environmental_hat = pickle.load(file)

with nutrient_model_path.open("rb") as file:
    loaded_nutrient_hat = pickle.load(file)


def predict_one_checked(model, reading, required_features):
    missing = [feature for feature in required_features if feature not in reading]
    if missing:
        raise ValueError(f"Missing features: {missing}")

    x = {feature: float(reading[feature]) for feature in required_features}

    return {
        "prediction": model.predict_one(x),
        "probabilities": model.predict_proba_one(x),
    }


environmental_example = X_env.iloc[0].to_dict()
nutrient_example = X_nutrient.iloc[0].to_dict()

print("Environmental example:")
print(predict_one_checked(loaded_environmental_hat, environmental_example, ENV_FEATURES))

print("\nNutrient-inclusive example:")
print(predict_one_checked(loaded_nutrient_hat, nutrient_example, NUTRIENT_FEATURES))


## Files to copy to the Raspberry Pi

After the notebook finishes, copy this folder from Google Drive:

`smart_farming_project/raspberry_pi_hat_models/`

It will contain:

- `environmental_health_hat.pkl`
- `nutrient_stress_hat.pkl`
- `hat_model_metadata.json`

The communication code should pass a sensor dictionary to each model. Only update a HAT with `learn_one()` after a trustworthy ground-truth label becomes available. Raw sensor readings by themselves are not labels.
